# Outlier detection

_Part of the **Decay Before Archival** project — see `README.md` for full project context. Reads the merged table written by `integrate_datasets.ipynb`; no BigQuery queries._

**Purpose:** Mid-Report 2, **Section 3: Outlier Detection**. Clustering (Section 1) and dimensionality reduction (Section 2) are owned by teammates.

The assignment asks for all of the following:

| # | Assignment item | Where in this notebook |
|---|---|---|
| a | Apply at least two outlier detection methods | §3 statistical baseline, §4 DBSCAN, §5 One-Class SVM, §6 GMM |
| b | Clearly define what an "outlier" is in our project context | §2 |
| c | Visualize detected outliers and explain their characteristics | §7, §8 |
| d | Discuss potential causes (data errors, rare but valid cases, automation, ...) | §9 |
| e | Evaluate the impact of outliers on clustering and dimensionality reduction | §10 |
| — | Feed Section 4 (integration, insights, limitations) | §12 |

**Methods come from the course only:** Lesson 11 (statistical approach, DBSCAN noise points) and Lesson 12 (One-Class SVM, Gaussian Mixture Model). The VAE from Lesson 12.4 is left out (it needs PyTorch and is heavy for ~15 tabular features); this goes in limitations.

**Data:** Table A (`data/merged_dataset.csv`, 62,627 repos × 49 columns). All rows are used, but deps.dev and Scorecard value columns are 98–100% missing (structural: most repos are not packages), so they are only used as 0/1 coverage flags in the main analysis and as real features in the packaged-only pass (§10). `inactive_next_365d` is **not** a feature; it is used afterwards to interpret outliers.

## 0. Dependencies on teammates

Section 3 can start on its own; only §10 (impact on clustering / DR) depends on the other sections. To keep the three sections consistent:

- **Shared feature matrix** (defined in §1, to send to the clustering and DR owners): GH Archive 2024 activity counts (log1p) + `days_since_last_push_at_cutoff` + coverage flags `in_depsdev`, `has_package`, `in_scorecard`, then `StandardScaler`.
- **Open questions for teammates:**
  - [ ] Clustering: which algorithms, which `k` / `eps` / `minPts`, same features and scaling?
  - [ ] DR: PCA components kept, t-SNE / UMAP params, same features and scaling?
  - [ ] Do they want the outlier flags (column `is_outlier`) exported so they can rerun with outliers removed?
  - [ ] If the clustering owner also uses DBSCAN, agree on `eps` / `minPts` so §4 and their section don't contradict each other (their noise points = our DBSCAN outliers).
- Until answers come back, §10 uses a **baseline K-means + PCA** with the settings in the `K` / `FEATURES` block, which can be swapped for theirs.

**Notes:**

- **Assignment item:** (coordination) dependencies and shared settings
- **What it shows:** _TODO_
- **Notes for the report:** _TODO_


## 1. Setup and feature matrix

- Load `merged_dataset.csv`.
- Drop identifiers (`repo_id`, `repo_full_name`, `repo_key`, `cutoff_date`), redundant H1/H2 splits and `*_missing` flags that mirror `has_*`.
- Exclude the sparse deps.dev / Scorecard value columns (keep the 0/1 coverage flags).
- `log1p` the skewed counts, fill `days_since_last_push_at_cutoff` NaN with 366 ("no push in window"), then `StandardScaler`.
- Keep `inactive_next_365d` aside for interpretation only.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA = Path("../data")
FIG_DIR = Path("../figures")
REPORT_IMG = Path("../report/img")

def save_fig(fig, name):
    """Write a figure to figures/ as SVG and report/img/ as PNG (no timestamp, so re-runs give clean diffs)."""
    fig.savefig(FIG_DIR / f"{name}.svg", bbox_inches="tight", metadata={"Date": None})
    fig.savefig(REPORT_IMG / f"{name}.png", bbox_inches="tight", dpi=150)

A = pd.read_csv(DATA / "merged_dataset.csv")
print(f"Table A: {len(A):,} rows x {A.shape[1]} cols")

In [ ]:
# TODO: build FEATURES list, X (log1p + fill + StandardScaler), y = A["inactive_next_365d"]
# Sanity checks: no NaN in X, label not in X

**Notes:**

- **Assignment item:** Outlier Detection → data used; why sparse deps.dev / Scorecard columns are excluded
- **What it shows:** _TODO_
- **Notes for the report:** _TODO_


## 2. What counts as an outlier here

- The typical repo is tiny: median 1 actor, 1 active month, 3 pushes in 2024.
- Draft definition: a repo whose 2024 activity profile is far from that bulk. Expected kinds:
  - **extreme automation**: huge push counts with a single actor (bots, mirrors, CI);
  - **unusually broad community**: many actors, PRs, reviews (rare but valid, popular projects);
  - **data artifacts**: duplicated names / recreated repos, undercounted events.
- Type of outlier (Lesson 11.2): mostly **global** outliers; note whether any are **contextual** (normal overall but odd for packaged repos, see §11).
- State the expected outlier share (~1%, used for One-Class SVM `nu` and the GMM likelihood cutoff) and why.

**Notes:**

- **Assignment item:** Outlier Detection → clearly define what constitutes an outlier in your project context
- **What it shows:** _TODO_
- **Notes for the report:** _TODO_


## 3. Baseline: statistical threshold (Lesson 11.3)

- Robust z = (x − median) / (1.4826 · MAD) on each log feature (IQR fallback for features with MAD = 0).
- Flag a repo if any feature has |robust z| > 3.5.
- Simple, explainable baseline: you can say *which* feature made it an outlier.

In [ ]:
# TODO: robust z-scores per feature, flag_stat, count + which features trigger most

**Notes:**

- **Assignment item:** Outlier Detection → statistical baseline (supports the outlier definition)
- **What it shows:** _TODO_
- **Notes for the report:** _TODO_


## 4. Method 1: DBSCAN noise points (Lesson 11.4, distance-based)

- `sklearn.cluster.DBSCAN` on `X`; points labelled `-1` (noise) are the outliers.
- `minPts` (`min_samples`) ≈ 2 × number of features.
- `eps` from the k-distance plot (sorted distance to the `minPts`-th neighbour via `NearestNeighbors`), taken at the knee.
- Report how the noise count changes with `eps` (small sensitivity table).

In [ ]:
# TODO: k-distance plot to choose eps

In [ ]:
# TODO: DBSCAN fit, flag_dbscan = labels == -1, eps sensitivity table

**Notes:**

- **Assignment item:** Outlier Detection → apply at least two methods (method 1: DBSCAN, distance-based)
- **What it shows:** _TODO_
- **Notes for the report:** _TODO_


## 5. Method 2: One-Class SVM (Lesson 12.2, classification-based)

- `sklearn.svm.OneClassSVM(kernel="rbf", nu=0.01, gamma="scale")`.
- Training cost grows ~quadratically with rows, so fit on a random 10k-row sample (`random_state=42`), then `decision_function` on all 62,627 repos.
- `nu` ≈ upper bound on the outlier fraction; justify 1%. Decision-score histogram with the cutoff marked.

In [ ]:
# TODO: OneClassSVM fit on sample, score all rows, flag_ocsvm, score histogram

**Notes:**

- **Assignment item:** Outlier Detection → apply at least two methods (method 2: One-Class SVM)
- **What it shows:** _TODO_
- **Notes for the report:** _TODO_


## 6. Method 3: Gaussian Mixture Model (Lesson 12.3, probability-based)

- `sklearn.mixture.GaussianMixture`, number of components chosen by BIC (e.g. 1–10).
- `reg_covar` raised a little: many features are discrete / binary, which can make covariances degenerate.
- Outlier = log-likelihood (`score_samples`) in the bottom 1%. Log-likelihood histogram with the cutoff marked.

In [ ]:
# TODO: BIC over n_components (plot)

In [ ]:
# TODO: GMM fit, score_samples, flag_gmm = bottom 1%

**Notes:**

- **Assignment item:** Outlier Detection → apply at least two methods (method 3: GMM)
- **What it shows:** _TODO_
- **Notes for the report:** _TODO_


## 7. Agreement between methods

- Overlap counts (Venn-style table) and Jaccard similarity between each pair of DBSCAN / One-Class SVM / GMM (and the statistical baseline).
- Consensus outliers = flagged by ≥ 2 of the 3 course methods.
- Top 15 consensus outliers with repo name and key counts.

In [ ]:
# TODO: overlap table, Jaccard, consensus flag, top-15 table

**Notes:**

- **Assignment item:** Outlier Detection → compare methods; which repos are robustly anomalous
- **What it shows:** _TODO_
- **Notes for the report:** _TODO_


## 8. Visualizing the outliers

- PCA 2-D scatter of `X`, outliers coloured by method.
- Box / strip plots of key features: inliers vs outliers.
- `inactive_next_365d` rate and `has_package` rate: inliers vs outliers (do outliers decay more or less than the 85% base rate?).

In [ ]:
# TODO: PCA scatter with outliers highlighted

In [ ]:
# TODO: feature box plots + label / coverage rates, inliers vs outliers

**Notes:**

- **Assignment item:** Outlier Detection → visualize detected outliers and explain their characteristics
- **What it shows:** _TODO_
- **Notes for the report:** _TODO_


## 9. Potential causes

- Inspect the top outliers by name.
- Tag each as: **automation / bot**, **mirror / generated**, **rare but valid (popular project)**, **data artifact** (duplicate name, collection gap).
- Count per category.

In [ ]:
# TODO: top outliers with names + hand-assigned cause category

**Notes:**

- **Assignment item:** Outlier Detection → discuss potential causes of these anomalies
- **What it shows:** _TODO_
- **Notes for the report:** _TODO_


## 10. Impact on clustering and dimensionality reduction

- Baseline (swap in teammates' settings when known): K-means with `K` chosen by silhouette on a sample, PCA on `FEATURES`.
- Fit **with** and **without** consensus outliers; compare:
  - PCA explained variance (PC1, PC2) and loadings;
  - cluster sizes, silhouette, centroid shifts;
  - Adjusted Rand Index on the shared inliers.
- Before / after figures.

In [ ]:
# === Settings to replace with teammates' choices ===
K = None          # TODO: number of clusters (from silhouette or clustering owner)
N_COMPONENTS = 2  # PCA components for plotting
# ====================================================
# TODO: fit K-means + PCA with / without outliers, compare metrics

In [ ]:
# TODO: before / after figures

**Notes:**

- **Assignment item:** Outlier Detection → evaluate the impact of outliers on clustering and dimensionality reduction
- **What it shows:** _TODO_
- **Notes for the report:** _TODO_


## 11. Packaged repos only (deps.dev features)

- The 951 repos with `has_package = 1`, same split as `pattern_mining.ipynb` §4.
- Add log1p deps.dev features (`n_versions_total`, `versions_2024`, `n_packages`, `stars_end`, `forks_end`).
- Rerun the same three methods (One-Class SVM can use all 951 rows here); compare with the full-table consensus (contextual outliers).

In [ ]:
# TODO: packaged subset, DBSCAN + One-Class SVM + GMM, overlap with §7 consensus

**Notes:**

- **Assignment item:** Outlier Detection → secondary analysis on the subset where deps.dev data exists
- **What it shows:** _TODO_
- **Notes for the report:** _TODO_


## 12. Summary for the report

Report-ready bullets for Section 3, plus inputs for Section 4 (integration, insights, limitations, next steps).

**Notes:**

- **Outlier definition:** _TODO_
- **Methods and agreement:** _TODO_
- **Characteristics and causes:** _TODO_
- **Impact on clustering / DR:** _TODO_
- **Actionable insight:** _TODO_
- **Limitations / next steps:** _TODO_
